In [26]:
import sys, pandas
print(sys.executable)

c:\Users\USER\Bitget-Report-2\.venv\Scripts\python.exe


In [27]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import numpy as np
import nbformat
from plotly.subplots import make_subplots

import plotly.io._renderers as _pr
_pr.nbformat = nbformat

In [28]:
from plotting_utils import DEFAULT_THEME, update_timeseries_layout, add_watermark

In [29]:
# ── Load data from Excel ──────────────────────────────────────────────
EXCEL_PATH = "data/Macro-analysis-data.xlsx"

# "corr" sheet: daily BTC, SPX, NDX, XAU, DXY (2015–2026)
df_corr = pd.read_excel(EXCEL_PATH, sheet_name="corr", parse_dates=["Date"]).set_index("Date")

# "spread" sheet: yields, DXY, BTC, SPX, NDX (2015–2026)
df_spread = pd.read_excel(EXCEL_PATH, sheet_name="spread", parse_dates=["Date"]).set_index("Date")

# "GOLD_ETFs" sheet: XAU and XAG spot prices (2000–2026)
df_metals = pd.read_excel(EXCEL_PATH, sheet_name="GOLD_ETFs", parse_dates=["Date"]).set_index("Date")

# ── Build a single clean daily DataFrame for the main assets ──────────
df = pd.DataFrame({
    "BTC":  df_corr["BTC Spot"],
    "SPX":  df_corr["SPX"],
    "NDX":  df_corr["NDX"],
    "XAU":  df_metals["XAUUSD BGN Curncy  (R2)"],
    "XAG":  df_metals["XAG BGN Curncy  (R1)"],
    "DXY":  df_corr["DXY"],
}).sort_index()

print(f"Combined DataFrame: {len(df)} rows, {df.index.min().date()} → {df.index.max().date()}")
print(df.dropna(subset=["BTC"]).tail(3))

Combined DataFrame: 7790 rows, 2000-01-03 → 2026-03-04
                 BTC      SPX       NDX      XAU      XAG     DXY
Date                                                             
2026-03-02  69423.73  6881.62  24992.60  5322.12  89.3780  98.381
2026-03-03  68033.05  6816.63  24720.08  5088.83  82.0205  99.052
2026-03-04  71803.31      NaN       NaN  5170.27  84.6250  98.895


In [30]:
# ── Colour palette ────────────────────────────────────────────────────
COLOURS = {
    "BTC":  "#F7931A",   # bitcoin orange
    "SPX":  "red",   # blue
    "NDX":  "white",   # pink
    "XAU":  "#FFD700",   # gold
    "XAG":  "#C0C0C0",   # silver
    "DXY":  "#2ca02c",   # green
    "10Y":  "#d62728",   # red
    "2Y":   "#9467bd",   # purple
    "CURVE":"#17becf",   # teal
}

# ── Reusable dual-axis chart builder ─────────────────────────────────
def dual_axis_chart(
    data: pd.DataFrame,
    left_col: str,
    right_col: str,
    left_label: str  = None,
    right_label: str = None,
    left_color: str  = None,
    right_color: str = None,
    left_prefix: str = "$",
    right_prefix: str = "",
    start_date: str  = None,
    event_dates: list = None,
):
    """Plot two series on a shared time axis with independent y-axes."""
    left_label  = left_label  or left_col
    right_label = right_label or right_col
    left_color  = left_color  or COLOURS.get(left_col)
    right_color = right_color or COLOURS.get(right_col)

    plot_df = data[[left_col, right_col]].copy()
    if start_date:
        plot_df = plot_df.loc[start_date:]

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=plot_df.index, y=plot_df[left_col],
        mode="lines", name=left_label,
        line=dict(color=left_color, width=2),
        yaxis="y1",
    ))
    fig.add_trace(go.Scatter(
        x=plot_df.index, y=plot_df[right_col],
        mode="lines", name=right_label,
        line=dict(color=right_color, width=2),
        yaxis="y2",
    ))

    fig.update_layout(
        showlegend=True,
        title="",
        height=650, width=1450,
        font=dict(size=20, color="white"),
        plot_bgcolor="#101A2E",
        paper_bgcolor="#101A2E",
        hovermode="x unified",
        xaxis=dict(
            title="", automargin=True, tickangle=0,
            showgrid=True, gridwidth=1, gridcolor="#293e68",
            zeroline=False, showline=False, ticklen=10,
        ),
        yaxis1=dict(
            title=left_label, gridcolor="#293e68",
            zeroline=False, showgrid=True, tickprefix=left_prefix,
        ),
        yaxis2=dict(
            title=right_label, overlaying="y", side="right",
            zeroline=False, showgrid=False, tickprefix=right_prefix,
        ),
        legend=dict(
            orientation="h", yanchor="top", y=1.1, xanchor="center", x=0.5,
        ),
        margin=dict(l=150, r=30, b=50, t=50, pad=4),
    )

    # Optional vertical event-date lines
    for d in (event_dates or []):
        fig.add_shape(
            type="line", x0=d, x1=d, y0=0, y1=1,
            xref="x", yref="paper",
            line=dict(color="white", width=2, dash="dash"),
        )

    add_watermark(fig)
    return fig

In [33]:
# ── Chart 1: BTC vs S&P 500 ───────────────────────────────────────────
fig1 = dual_axis_chart(
    df,
    left_col="BTC",   right_col="SPX",
    left_label="BTC Spot",
    right_label="S&P 500",
    left_prefix="$", right_prefix="",
    start_date="2020-01-01",
)
fig1

In [36]:
# ── Chart 2: BTC vs Nasdaq 100 ────────────────────────────────────────
fig2 = dual_axis_chart(
    df_spread,
    left_col="BTC Spot Price",   right_col="NDX",
    left_label="BTC Spot",
    right_label="Nasdaq 100",
    left_prefix="$", right_prefix="",
    start_date="2020-01-01",
)
fig2

In [37]:
# ── Chart 3: BTC vs Gold ──────────────────────────────────────────────
fig3 = dual_axis_chart(
    df,
    left_col="BTC",   right_col="XAU",
    left_label="BTC Spot",
    right_label="Gold Spot (USD/oz)",
    left_prefix="$", right_prefix="$",
    start_date="2020-01-01",
)
fig3

In [38]:
df_spread

,US 10 Year Yield,US 30 Year Yield,China 10 Year Yield,US Minus China 10Y Spread,BTC Spot Price,US 2 Year Yield,China 2 Year Yield,2s10s,DXY,BTC ReT,Japan 10Y,Japan 2Y,SPX,NDX,US 5 Year Yield,5s30s,10Y US Minus 10Y Japan
Date,,,,,,,,,,,,,,,,,
2015-01-01,NaN,NaN,NaN,NaN,314.49,NaN,NaN,NaN,90.276,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-02,2.1105,2.6880,NaN,NaN,314.73,0.6647,NaN,1.4458,91.080,0.000763,NaN,NaN,2058.20,4230.2402,1.6070,1.0810,NaN
2015-01-04,NaN,NaN,NaN,NaN,266.26,NaN,NaN,NaN,NaN,-0.154005,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-05,2.0320,2.5987,3.6500,-1.6180,270.93,0.6568,3.3400,1.3752,91.378,0.017539,0.3200,-0.0250,2020.58,4160.9600,1.5645,1.0342,1.7120
2015-01-06,1.9402,2.5023,3.6400,-1.6998,286.41,0.6250,3.3300,1.3152,91.499,0.057137,0.2880,-0.0260,2002.61,4110.8301,1.4780,1.0243,1.6522
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-02-28,NaN,NaN,NaN,NaN,66723.70,NaN,NaN,NaN,NaN,0.018257,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-03-01,NaN,NaN,NaN,NaN,65678.10,NaN,NaN,NaN,NaN,-0.015671,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-03-02,4.0345,4.6797,1.8012,2.2333,69423.73,3.4752,1.3554,0.5593,98.381,0.057030,2.0765,1.2205,6881.62,24992.6000,3.6103,1.0694,1.9580


In [39]:
fig = dual_axis_chart(
    df_spread,
    left_col="BTC Spot Price",   right_col="2s10s",
    left_label="BTC Spot",
    right_label="2s10s",
    left_prefix="$", #right_suffix="%",
    start_date="2020-01-01",
)
fig

In [40]:
# ── Correlations from "corr" sheet: BTC vs SPX, NDX, XAU ─────────────
# ↓↓ CHANGE THIS to 30 / 60 / 90 etc. and re-run ↓↓
correlation_window = 90
w = correlation_window

# Returns
df_corr['BTC Daily Return'] = df_corr['BTC Spot'].pct_change()
df_corr['SPX Daily Return'] = df_corr['SPX'].pct_change()
df_corr['NDX Daily Return'] = df_corr['NDX'].pct_change()
df_corr['XAU Daily Return'] = df_corr['XAU'].pct_change()

# Drop rows with missing returns
df_corr = df_corr.dropna(subset=[
    'BTC Daily Return', 'SPX Daily Return', 'NDX Daily Return', 'XAU Daily Return'
])

# Rolling correlations
df_corr[f'BTC_SPX_corr_{w}d'] = df_corr['BTC Daily Return'].rolling(window=w).corr(df_corr['SPX Daily Return'])
df_corr[f'BTC_NDX_corr_{w}d'] = df_corr['BTC Daily Return'].rolling(window=w).corr(df_corr['NDX Daily Return'])
df_corr[f'BTC_XAU_corr_{w}d'] = df_corr['BTC Daily Return'].rolling(window=w).corr(df_corr['XAU Daily Return'])

print(df_corr[[f'BTC_SPX_corr_{w}d', f'BTC_NDX_corr_{w}d', f'BTC_XAU_corr_{w}d']].dropna().tail(3))

            BTC_SPX_corr_90d  BTC_NDX_corr_90d  BTC_XAU_corr_90d
Date                                                            
2026-03-02          0.547942          0.520830          0.200047
2026-03-03          0.547281          0.523026          0.211562
2026-03-04          0.537385          0.512901          0.222134


C:\Users\USER\AppData\Local\Temp\ipykernel_22704\4199693064.py:8: FutureWarning:

The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.

C:\Users\USER\AppData\Local\Temp\ipykernel_22704\4199693064.py:9: FutureWarning:

The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.

C:\Users\USER\AppData\Local\Temp\ipykernel_22704\4199693064.py:10: FutureWarning:

The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.



In [41]:
# ── Chart 4/5: BTC rolling correlation vs SPX, NDX, Gold ─────────────
w = correlation_window

fig4 = go.Figure()

for col, label, color in [
    (f'BTC_SPX_corr_{w}d', 'SPX', COLOURS['SPX']),
    (f'BTC_NDX_corr_{w}d', 'NDX', COLOURS['NDX']),
    (f'BTC_XAU_corr_{w}d', 'XAU', COLOURS['XAU']),
]:
    series = df_corr[col].dropna()
    fig4.add_trace(go.Scatter(
        x=series.index, y=series.values,
        mode='lines', name=label,
        line=dict(color=color, width=2),
    ))

fig4.add_hline(y=0, line_dash="dot", line_color="white", opacity=0.4)

fig4.update_layout(
    showlegend=True, title="", height=650, width=1450,
    font=dict(size=20, color="white"),
    plot_bgcolor="#101A2E", paper_bgcolor="#101A2E",
    hovermode="x unified",
    xaxis=dict(title="", automargin=True, tickangle=0,
               showgrid=True, gridwidth=1, gridcolor="#293e68",
               zeroline=False, showline=False, ticklen=10),
    yaxis=dict(title=f"BTC {w}d Rolling Correlation",
               gridcolor="#293e68", zeroline=False, showgrid=True),
    legend=dict(orientation="h", yanchor="top", y=1.1, xanchor="center", x=0.5),
    margin=dict(l=150, r=30, b=50, t=50, pad=4),
)
add_watermark(fig4)
fig4

In [42]:
# ── Correlations from "spread" sheet: BTC vs yields & DXY ────────────
w = correlation_window

df_spread['BTC Daily Return'] = df_spread['BTC Spot Price'].pct_change()
df_spread['DXY Daily Return'] = df_spread['DXY'].pct_change()
df_spread['10Y Change']   = df_spread['US 10 Year Yield'].diff()
df_spread['2Y Change']    = df_spread['US 2 Year Yield'].diff()
df_spread['2s10s Change'] = df_spread['2s10s'].diff()

# Drop rows with missing returns/changes
df_spread = df_spread.dropna(subset=[
    'BTC Daily Return', 'DXY Daily Return', '10Y Change', '2Y Change', '2s10s Change'
])

df_spread[f'BTC_10Y_corr_{w}d']   = df_spread['BTC Daily Return'].rolling(window=w).corr(df_spread['10Y Change'])
df_spread[f'BTC_2Y_corr_{w}d']    = df_spread['BTC Daily Return'].rolling(window=w).corr(df_spread['2Y Change'])
df_spread[f'BTC_2s10s_corr_{w}d'] = df_spread['BTC Daily Return'].rolling(window=w).corr(df_spread['2s10s Change'])
df_spread[f'BTC_DXY_corr_{w}d']   = df_spread['BTC Daily Return'].rolling(window=w).corr(df_spread['DXY Daily Return'])

print(df_spread[[f'BTC_10Y_corr_{w}d', f'BTC_2Y_corr_{w}d', f'BTC_2s10s_corr_{w}d', f'BTC_DXY_corr_{w}d']].dropna().tail(3))

# ── Chart 6: BTC rolling correlation vs US yields ────────────────────
fig6 = go.Figure()

for col, label, color in [
    (f'BTC_10Y_corr_{w}d',   '10Y Yield Chg',  COLOURS['10Y']),
    (f'BTC_2Y_corr_{w}d',    '2Y Yield Chg',   COLOURS['2Y']),
    (f'BTC_2s10s_corr_{w}d', '2s10s Curve Chg', COLOURS['CURVE']),
    (f'BTC_DXY_corr_{w}d',   'DXY Daily Return', COLOURS['DXY']),
]:
    series = df_spread[col].dropna()
    fig6.add_trace(go.Scatter(
        x=series.index, y=series.values,
        mode='lines', name=label,
        line=dict(color=color, width=2),
    ))

fig6.add_hline(y=0, line_dash="dot", line_color="white", opacity=0.4)

fig6.update_layout(
    showlegend=True, title="", height=650, width=1450,
    font=dict(size=20, color="white"),
    plot_bgcolor="#101A2E", paper_bgcolor="#101A2E",
    hovermode="x unified",
    xaxis=dict(title="", automargin=True, tickangle=0,
               showgrid=True, gridwidth=1, gridcolor="#293e68",
               zeroline=False, showline=False, ticklen=10),
    yaxis=dict(title=f"BTC {w}d Rolling Correlation",
               gridcolor="#293e68", zeroline=False, showgrid=True),
    legend=dict(orientation="h", yanchor="top", y=1.1, xanchor="center", x=0.5),
    margin=dict(l=150, r=30, b=50, t=50, pad=4),
)
add_watermark(fig6)
fig6

            BTC_10Y_corr_90d  BTC_2Y_corr_90d  BTC_2s10s_corr_90d  \
Date                                                                
2026-02-27          0.234610         0.239752           -0.003760   
2026-03-03          0.248680         0.249970            0.000933   
2026-03-04          0.254685         0.251228            0.016774   

            BTC_DXY_corr_90d  
Date                          
2026-02-27         -0.131644  
2026-03-03         -0.132613  
2026-03-04         -0.144298  


C:\Users\USER\AppData\Local\Temp\ipykernel_22704\1279585785.py:5: FutureWarning:

The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.

C:\Users\USER\AppData\Local\Temp\ipykernel_22704\1279585785.py:15: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\USER\AppData\Local\Temp\ipykernel_22704\1279585785.py:16: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returni

In [43]:
# ── Chart 7: Scatter plot with OLS regression ────────────────────────
# ↓↓ CONFIGURE THESE ↓↓
x_col       = "SPX"           # change to "NDX", "XAU", etc.
y_col       = "BTC"           # change to any column in df
start_date  = "2024-01-01"    # lookback start
# ↑↑ CONFIGURE THESE ↑↑

import statsmodels.api as sm
import plotly.express as px

# Filter and prepare data
scatter_df = df[[x_col, y_col]].loc[start_date:].dropna().copy()

# OLS regression
X = sm.add_constant(scatter_df[x_col])
model = sm.OLS(scatter_df[y_col], X).fit()
scatter_df["Regression_Line"] = model.predict(X)
regression_label = f"R² = {model.rsquared:.4f}"

# Month labels for colour axis
scatter_df["MonthLabel"] = scatter_df.index.strftime("%b %y")
month_order = pd.date_range(
    start=start_date,
    end=scatter_df.index.max(),
    freq="MS"
).strftime("%b %y").tolist()
month_to_num = {m: i for i, m in enumerate(month_order)}
scatter_df["MonthNum"] = scatter_df["MonthLabel"].map(month_to_num)

# Scatter
fig7 = px.scatter(
    scatter_df,
    x=x_col, y=y_col,
    color="MonthNum",
    color_continuous_scale="Peach",
    labels={"MonthNum": "Month"},
)

# Regression line
fig7.add_trace(go.Scatter(
    x=scatter_df[x_col],
    y=scatter_df["Regression_Line"],
    mode="lines", name=regression_label,
    line=dict(color="white", width=2),
))

# Highlight latest point
latest = scatter_df.iloc[-1]
fig7.add_trace(go.Scatter(
    x=[latest[x_col]], y=[latest[y_col]],
    mode="markers", name="Latest",
    marker=dict(color="#14F195", size=15, symbol="x"),
    showlegend=True,
))

# Layout
fig7.update_layout(
    height=1000, width=1000,
    font=dict(size=20, color="white"),
    plot_bgcolor="#101A2E", paper_bgcolor="#101A2E",
    hovermode="closest", title="",
    xaxis=dict(title=x_col, showgrid=True, gridcolor="#293e68",
               zeroline=False, tickformat="~s"),
    yaxis=dict(title=y_col, tickprefix="$", showgrid=True,
               gridcolor="#293e68", zeroline=False),
    legend=dict(orientation="h", yanchor="top", y=1.1,
                xanchor="center", x=0.5),
    margin=dict(l=150, r=30, b=50, t=80, pad=4),
    coloraxis_colorbar=dict(
        title="",
        tickvals=list(month_to_num.values()),
        ticktext=list(month_to_num.keys()),
    ),
)
add_watermark(fig7)
fig7

In [44]:
# ── Chart 7b: Multi-regime scatter with separate OLS ─────────────────

x_col, y_col = "SPX", "BTC"

# ── Define regimes ───────────────────────────────────────────────────
regimes = {
    "Jan 24 - Oct 10 25":  ("2024-01-01", "2025-10-10"),
    "Oct 10 25 - Jan 1 26": ("2025-10-10", "2026-01-01"),
    "Jan 1 26 - Present":   ("2026-01-01", None),
}
regime_colours = {
    "Jan 24 - Oct 10 25":   "#17becf",   # teal
    "Oct 10 25 - Jan 1 26": "#FFD700",   # gold
    "Jan 1 26 - Present":   "#d62728",   # red
}

fig7b = go.Figure()

for regime_name, (r_start, r_end) in regimes.items():
    # Slice regime
    chunk = df[[x_col, y_col]].loc[r_start:r_end].dropna()
    if r_end and r_end in chunk.index:
        chunk = chunk.loc[:pd.Timestamp(r_end) - pd.Timedelta(days=1)]
    if len(chunk) < 5:
        continue

    colour = regime_colours[regime_name]

    # OLS for this regime
    X = sm.add_constant(chunk[x_col])
    model = sm.OLS(chunk[y_col], X).fit()
    beta  = model.params[x_col]
    r2    = model.rsquared

    # Scatter points
    fig7b.add_trace(go.Scatter(
        x=chunk[x_col], y=chunk[y_col],
        mode="markers", name=f"{regime_name}",
        marker=dict(color=colour, size=6),
    ))
"""
    # Regression line
    x_range = pd.Series([chunk[x_col].min(), chunk[x_col].max()])
    X_line = sm.add_constant(x_range)
    fig7b.add_trace(go.Scatter(
        x=x_range, y=model.predict(X_line),
        mode="lines", name=f"R²={r2:.3f}",
        line=dict(color=colour, width=3),
    )) """

# Highlight latest point
latest = df[[x_col, y_col]].dropna().iloc[-1]
fig7b.add_trace(go.Scatter(
    x=[latest[x_col]], y=[latest[y_col]],
    mode="markers", name="Latest",
    marker=dict(color="#14F195", size=15, symbol="x"),
))

fig7b.update_layout(
    height=1000, width=1000,
    font=dict(size=20, color="white"),
    plot_bgcolor="#101A2E", paper_bgcolor="#101A2E",
    hovermode="closest", title="",
    xaxis=dict(title=x_col, showgrid=True, gridcolor="#293e68",
               zeroline=False, tickformat="~s"),
    yaxis=dict(title=y_col, tickprefix="$", showgrid=True,
               gridcolor="#293e68", zeroline=False),
    legend=dict(orientation="h", yanchor="top", y=1.12,
                xanchor="center", x=0.5),
    margin=dict(l=150, r=30, b=50, t=100, pad=4),
)
add_watermark(fig7b)
fig7b

In [ ]:
# ── Chart 7c: Rolling beta & R² (BTC ~ SPX) with regime boundaries ───
import statsmodels.api as sm

x_col, y_col = "SPX", "BTC"
roll_window = 60       # ← change to 30, 90, etc.
roll_start  = "2024-01-01"

# Daily returns
ret = df[[x_col, y_col]].loc[roll_start:].pct_change().dropna()

# Rolling cov / var for beta, rolling corr² for R²
rolling_cov = ret[y_col].rolling(roll_window).cov(ret[x_col])
rolling_var = ret[x_col].rolling(roll_window).var()
rolling_beta = rolling_cov / rolling_var
rolling_r2   = ret[y_col].rolling(roll_window).corr(ret[x_col]) ** 2

# Plot
fig7c = go.Figure()

# Beta on left axis
fig7c.add_trace(go.Scatter(
    x=rolling_beta.index, y=rolling_beta.values,
    mode="lines", name=f"{roll_window}d Rolling β",
    line=dict(color="#17becf", width=2),
    yaxis="y1",
))

# R² on right axis
fig7c.add_trace(go.Scatter(
    x=rolling_r2.index, y=rolling_r2.values,
    mode="lines", name=f"{roll_window}d Rolling R²",
    line=dict(color="#FFD700", width=2),
    yaxis="y2",
))
"""
# Regime boundary lines
for d, label in [("2025-10-10", "10/10 liquidation"), ("2026-01-01", "New Year 2026")]:
    fig7c.add_shape(
        type="line", x0=d, x1=d, y0=0, y1=1,
        xref="x", yref="paper",
        line=dict(color="white", width=2, dash="dash"),
    )
    fig7c.add_annotation(
        x=d, y=1.05, xref="x", yref="paper",
        text=label, showarrow=False,
        font=dict(color="white", size=14),
    )"""

fig7c.update_layout(
    height=650, width=1450,
    font=dict(size=20, color="white"),
    plot_bgcolor="#101A2E", paper_bgcolor="#101A2E",
    hovermode="x unified", title="",
    xaxis=dict(title="", automargin=True, tickangle=0,
               showgrid=True, gridwidth=1, gridcolor="#293e68",
               zeroline=False, showline=False, ticklen=10),
    yaxis1=dict(title=f"{roll_window}d Rolling Beta (BTC ~ SPX)",
                gridcolor="#293e68", zeroline=False, showgrid=True),
    yaxis2=dict(title=f"{roll_window}d Rolling R²",
                overlaying="y", side="right",
                zeroline=False, showgrid=False,
                range=[0, 1]),
    legend=dict(orientation="h", yanchor="top", y=1.1,
                xanchor="center", x=0.5),
    margin=dict(l=150, r=100, b=50, t=80, pad=4),
)
add_watermark(fig7c)
fig7c

C:\Users\USER\AppData\Local\Temp\ipykernel_22704\3998424874.py:9: FutureWarning:

The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.



In [9]:
# ── Regime summary stats ──────────────────────────────────────────────
import statsmodels.api as sm

x_col, y_col = "SPX", "BTC"
regimes = {
    "Jan 24 → Oct 10 25":   ("2024-01-01", "2025-10-10"),
    "Oct 10 25 → Jan 1 26": ("2025-10-10", "2026-01-01"),
    "Jan 1 26 → Present":   ("2026-01-01", None),
}

rows = []
for name, (r_start, r_end) in regimes.items():
    chunk = df[[x_col, y_col]].loc[r_start:r_end].dropna()
    if r_end:
        chunk = chunk.loc[:pd.Timestamp(r_end) - pd.Timedelta(days=1)]
    ret_chunk = chunk.pct_change().dropna()
    X = sm.add_constant(ret_chunk[x_col])
    m = sm.OLS(ret_chunk[y_col], X).fit()
    rows.append({
        "Regime": name,
        "N (days)": len(ret_chunk),
        "Beta": round(m.params[x_col], 3),
        "R²": round(m.rsquared, 4),
        "Corr": round(ret_chunk[x_col].corr(ret_chunk[y_col]), 4),
        "BTC ann. vol": f"{ret_chunk[y_col].std() * (252**0.5) * 100:.1f}%",
        "SPX ann. vol": f"{ret_chunk[x_col].std() * (252**0.5) * 100:.1f}%",
        "BTC total %": f"{(chunk[y_col].iloc[-1] / chunk[y_col].iloc[0] - 1) * 100:.1f}%",
        "SPX total %": f"{(chunk[x_col].iloc[-1] / chunk[x_col].iloc[0] - 1) * 100:.1f}%",
    })
pd.DataFrame(rows)

,Regime,N (days),Beta,R²,Corr,BTC ann. vol,SPX ann. vol,BTC total %,SPX total %
0,Jan 24 → Oct 10 25,444,1.113,0.1376,0.3709,48.9%,16.3%,168.5%,42.0%
1,Oct 10 25 → Jan 1 26,56,1.994,0.3356,0.5793,41.3%,12.0%,-23.3%,4.5%
2,Jan 1 26 → Present,40,3.299,0.3554,0.5962,66.3%,12.0%,-24.4%,-0.6%


In [27]:
# ── Load hourly macro data ────────────────────────────────────────────
HOURLY_PATH = "data/bitget__hourly_macro.xlsx"
df_hourly = pd.read_excel(HOURLY_PATH, sheet_name="Hourly 1yr", parse_dates=["Date"]).set_index("Date").sort_index()
df_hourly = df_hourly.rename(columns={
    "Gold": "XAU", "Silver": "XAG",
    "S&P 500": "SPX", "Nasdaq 100": "NDX",
    "U.S. Dollar Index (DXY)": "DXY",
})
print(f"Hourly data: {len(df_hourly)} rows, {df_hourly.index.min()} → {df_hourly.index.max()}")
print(df_hourly.tail(3))

# ── Chart 8 · YTD spot returns (%) ───────────────────────────────────
ytd_start = "2026-01-01"
ytd_assets = ["BTC", "SPX", "NDX", "XAU", "XAG"]

ytd = df_hourly[ytd_assets].loc[ytd_start:].dropna(how="all")
ytd_base = ytd.bfill().iloc[0]
ytd_pct = (ytd / ytd_base - 1) * 100   # percentage return

fig8 = go.Figure()
for col in ytd_assets:
    fig8.add_trace(go.Scatter(
        x=ytd_pct.index, y=ytd_pct[col],
        mode="lines", name=col,
        line=dict(color=COLOURS.get(col, "white"), width=2),
    ))

fig8.add_hline(y=0, line=dict(color="white", width=1, dash="dot"))

update_timeseries_layout(fig8)
fig8.update_layout(
    yaxis=dict(title="YTD Return (%)", ticksuffix="%"),
)
add_watermark(fig8)
fig8

Hourly data: 8955 rows, 2025-02-06 22:00:00 → 2026-03-05 11:30:00
                          BTC       ETH      XAU      XAG  SPX  NDX     DXY
Date                                                                       
2026-03-05 09:30:00  73189.64  2144.417  5156.24  84.1060  NaN  NaN  98.915
2026-03-05 10:30:00  73144.38  2148.930  5163.20  84.2934  NaN  NaN  98.983
2026-03-05 11:30:00  72905.12  2130.940  5167.04  84.2747  NaN  NaN  98.978


In [28]:
# ── Chart 9 · Returns since Trump inauguration (%) ──────────────────
# Hourly data starts Feb 2025 — use daily sheet for longer lookback
df_daily = pd.read_excel(HOURLY_PATH, sheet_name="Daily 5y", parse_dates=["Date"]).set_index("Date").sort_index()
df_daily = df_daily.rename(columns={
    "Gold": "XAU", "Silver": "XAG",
    "S&P 500": "SPX", "Nasdaq 100": "NDX",
    "U.S. Dollar Index (DXY)": "DXY",
})

inaug_date = "2025-01-20"
inaug_assets = ["BTC", "SPX", "NDX", "XAU", "XAG", "DXY"]

inaug = df_daily[inaug_assets].loc[inaug_date:].dropna(how="all")
inaug_base = inaug.bfill().iloc[0]
inaug_pct = (inaug / inaug_base - 1) * 100

fig9 = go.Figure()
for col in inaug_assets:
    fig9.add_trace(go.Scatter(
        x=inaug_pct.index, y=inaug_pct[col],
        mode="lines", name=col,
        line=dict(color=COLOURS.get(col, "white"), width=2),
    ))

fig9.add_hline(y=0, line=dict(color="white", width=1, dash="dot"))

update_timeseries_layout(fig9)
fig9.update_layout(
    yaxis=dict(title="Return since Inauguration (%)", ticksuffix="%"),
)
add_watermark(fig9)
fig9